In [22]:
league_name = input("League name (NFL, MLB, NBA, NHL, or MLS): ")

In [23]:
from ipyleaflet import Map, Popup, Marker

import ipywidgets as widgets

def opacity_for_population(share_population_value): 
    if share_population_value > 5000000:
        return 0.9
    if share_population_value > 1000000:
        return 0.8
    if share_population_value > 500000:
        return 0.7  
    if share_population_value > 100000:
        return 0.6 
    if share_population_value > 50000:
        return 0.5
    if share_population_value > 10000:
        return 0.4
    if share_population_value > 5000:
        return 0.3
    if share_population_value > 1000:
        return 0.2
    return 0.1  

def reset_county_styles(counties_geojson):
    default_style = {
        "color": "grey",
        "weight": 1,
        "fillColor": "grey",
        "fillOpacity": 0.0,
    }

    for feature in counties_geojson["features"]:
        feature["properties"]["style"] = default_style

def heatmap_counties(counties_geojson, league_county_map: dict, share_threshold = 0.01): 
        reset_county_styles(counties_geojson)
        for feature in counties_geojson["features"]:
            geoid = feature["properties"]["geoid"]
            county_data = league_county_map[geoid]
            county_top_team_data = county_data["team_stats"][0]
            if county_top_team_data["share"] > share_threshold:   
                feature["properties"]["style"] = {
                    "color": "grey",
                    "weight": 1,
                    "fillColor": county_top_team_data["color"] ,
                    "fillOpacity": opacity_for_population(county_top_team_data["share_population_value"])
                }               

def create_show_teams(leaflet_map:Map, league_county_map: dict):
    def show_teams(event, feature, **kwargs): 
        coordinates = kwargs.get("coordinates")
        geoid = feature["properties"]["geoid"]
        county_data = league_county_map[geoid]
        county_team_data = county_data["team_stats"]
        leagues_rows = ""  
        for i, county_team_row in enumerate(county_team_data):
            if county_team_row["share"] > 1/len(county_team_data):
                leagues_rows += f'''
                    <tr>  
                        <td>{county_team_row['team_name']}</td> 
                        <td>{county_team_row['share_population']:,.0f}</td> 
                        <td>{county_team_row['share_population_value']:,.0f}</td> 
                    </tr>'''

        popup = Popup(location = coordinates, max_width=500,
            child=widgets.HTML(f'''
                <table style="border-collapse: collapse;">
                    <caption>{feature["properties"]["name"]} - Pop: {county_data["population"]:,.0f}</caption>
                    <tr>
                        <th>Team</th>
                        <th>Pop. Share</th>
                        <th>Pop. Value</th>
                    </tr>
                {leagues_rows}
                </table>'''))  
        leaflet_map.add(popup)
    return show_teams  

In [24]:
import polars as pl

def league_teams_sums(county_data) -> pl.DataFrame:
    all_teams = pl.DataFrame([x for county in county_data.values() for x in county["team_stats"]])

    return all_teams.select(['team_name', 'share_population', 'share_population_value'])\
            .group_by('team_name')\
            .agg(pl.sum('share_population'), pl.sum('share_population_value'))\
            .sort('share_population_value', descending=True)  

out = widgets.Output()

@out.capture()
def show_comparisons(league_calculations_prior, league_calculations_after):
    out.clear_output()
    before_sums = league_teams_sums(league_calculations_prior)\
        .rename({"share_population": "share_population_before", "share_population_value": "share_population_value_before"})

    after_sums = league_teams_sums(league_calculations_after)\
        .rename({"share_population": "share_population_after", "share_population_value": "share_population_value_after"})


    merged = before_sums.join(after_sums, on='team_name', how='full', coalesce=True)\
            .fill_null(0).with_columns([
               (pl.col("share_population_after") - pl.col("share_population_before")).alias("share_population_change"),
               (pl.col("share_population_value_after") - pl.col("share_population_value_before")).alias("share_population_value_change")
            ])
    
    pl.Config.set_tbl_rows(50)

    formatter = lambda x: f"{x:,.0f}"
    
    print(merged.with_columns([
        pl.col("share_population_before").map_elements(formatter).alias("share_population_before"),
        pl.col("share_population_after").map_elements(formatter).alias("share_population_after"),
        pl.col("share_population_value_before").map_elements(formatter).alias("share_population_value_before"),
        pl.col("share_population_value_after").map_elements(formatter).alias("share_population_value_after"),
        pl.col("share_population_change").map_elements(formatter).alias("share_population_change"),
        pl.col("share_population_value_change").map_elements(formatter).alias("share_population_value_change")
    ]))
    
    print(f'Total Population Values Sums\t: {merged["share_population_value_before"].sum():,.0f}\t{merged["share_population_value_after"].sum():,.0f}')
    print(f'Total Population Values Change = {(merged["share_population_value_after"].sum() - merged["share_population_value_before"].sum()):,.0f}')      

In [25]:
import json
import ipywidgets as widgets

from IPython.display import display
from ipyleaflet import Map, GeoJSON, FullScreenControl, WidgetControl, basemaps

import os

os.environ["RUST_BACKTRACE"] = "1"

import pyrust

with open(f'teams_{league_name}.json', "r") as f:
    league = json.load(f)

with open("./counties-4326.geojson") as f:
    counties_geojson =  json.load(f)   

leagues_data = { 

}

calculator = pyrust.LeagueStatsCalculator() 

result = calculator.load_league(league)

leagues_data[result["league_name"]] = result["county_stats_by_geoid"]

display(widgets.HTML("""
            <style>
            .leaflet-popup-content-wrapper .leaflet-popup-tip {
                background-color: black;
                border: 2px solid black;
            }
            .leaflet-popup-content {
               color: white;
               max-height: 350px;  
               overflow-y: auto; 
            }
            
            table {
                style="border-collapse: collapse;"
            }

            th,td {
                padding: 0 10px;
            }
            </style>"""))

heatmap_counties(counties_geojson, result["county_stats_by_geoid"])

map = Map(basemap=basemaps.CartoDB.Positron, center=[38.72728229549864, -96.9010842308538], zoom=5, scroll_wheel_zoom=True)

geojson_layer = GeoJSON(data = counties_geojson,  hover_style = {"color": "white"})

prior_league_calculations = result["county_stats_by_geoid"]

calc_button=widgets.Button(description='Re-Calculate')
def recalculate_stats(event):
    global leagues_data, counties_geojson, geojson_layer, map, league, calculator, calc_button
    try:
        calc_button.description = "Calculating..."
        calc_button.disabled = True
        map.interaction = False

        map.remove_layer(geojson_layer)

        result = calculator.load_league(league)
    
        leagues_data[result["league_name"]] = result["county_stats_by_geoid"]

        heatmap_counties(counties_geojson, result["county_stats_by_geoid"])
       
        geojson_layer = GeoJSON(data = counties_geojson,  hover_style = {"color": "white"})
  
        geojson_layer.on_click(create_show_teams(map, result["county_stats_by_geoid"]))
        map.add_layer(geojson_layer)

        show_comparisons(prior_league_calculations, result["county_stats_by_geoid"])
    finally:         
        calc_button.description = "Re-Calculate"
        calc_button.disabled = False
        map.interaction = True

def create_move_handler(team):
    @out.capture()
    def move_handler(event, **kwargs):
        team["coordinates"]["lat"] = kwargs["location"][0]
        team["coordinates"]["lon"] = kwargs["location"][1]
        team["state"] = calculator.lookup_state_name_by_coordinates(team["coordinates"]["lat"], team["coordinates"]["lon"])
    return move_handler

current_popup = None    

def create_popup_for_team(team, new_team=False):
    global league, map
    team_text = widgets.Text(value=team["name"])
    venue_text = widgets.Text(value=team["venue"])
    l_text = widgets.FloatText(value=team["L"])
    s_text = widgets.FloatText(value=team["S"])
    n_text = widgets.FloatText(value=team["N"])
    if type(team["state"]) is list:
        state_text = widgets.Text(value=", ".join(team["state"]))
    else:
        state_text = widgets.Text(value=team["state"])
    color_picker = widgets.ColorPicker(value=team["color"])
    update_button = widgets.Button(description="Update" if not new_team else "Create", layout=widgets.Layout(width="95%"))
    delete_button = widgets.Button(description="Delete")

    gridbox = widgets.GridBox(
        children=[
            widgets.Label("Team:"), team_text,
            widgets.Label("Venue:"), venue_text,
            widgets.Label("Coordinates:"), widgets.Label(f"({team['coordinates']['lat']}, {team['coordinates']['lon']})"),
            widgets.Label("L:"), l_text,
            widgets.Label("S:"), s_text,
            widgets.Label("N:"), n_text,
            widgets.Label("State:"), state_text,
            widgets.Label("Color:"), color_picker,
            update_button, delete_button if not new_team else widgets.Label("")
        ],
        layout=widgets.Layout(grid_template_columns="100px 1fr"))

    def on_update_click(x):
        global current_popup
        if len(team_text.value.strip()) > 0:
            team["name"] = team_text.value
        if len(venue_text.value.strip()) > 0:    
            team["venue"] = venue_text.value
        if l_text.value > 0:
            team["L"] = l_text.value
        if s_text.value > 0:
            team["S"] = s_text.value
        if n_text.value > 0:
            team["N"] = n_text.value
        if len(state_text.value.strip()) > 0:
            team["state"] = state_text.value.split(",")[0].strip() if "," in state_text.value else state_text.value.strip()
        if len(color_picker.value.strip()) > 0:
            team["color"] = color_picker.value   
        if new_team:
            league["teams"].append(team)
            map.add(create_marker_for_team(team))
        map.remove(current_popup) 
        current_popup = None

    def on_delete_click(x):
        global current_popup
        league["teams"] = [t for t in league["teams"] if t != team]
        map.remove(current_popup) 
        current_popup = None
        for layer in map.layers:
            if isinstance(layer, Marker) and layer.title == team["name"]:
                map.remove(layer)
                break    
        
    update_button.on_click(on_update_click)    
    delete_button.on_click(on_delete_click)
    return gridbox

def create_marker_click_handler(team):
    @out.capture()
    def on_marker_click(**kwargs):
        global current_popup

        if current_popup is not None:
            map.remove(current_popup)

        current_popup = Popup(location=kwargs.get("coordinates"), min_width=420, max_width=420)
        current_popup.child = create_popup_for_team(team)
        map.add(current_popup)
    return on_marker_click    

def create_marker_for_team(team):
    marker = Marker(
            location=(team["coordinates"]["lat"], team["coordinates"]["lon"]))
    marker.on_move(create_move_handler(team))
    marker.on_click(create_marker_click_handler(team))
    return marker

@out.capture()
def handle_interaction(**kwargs):
    global current_popup, map

    if kwargs.get("type") != "contextmenu":
        return    

    content = create_popup_for_team({
            "name": "New Team",
            "venue": "Stadium Arena",
            "coordinates": {"lat": kwargs.get("coordinates")[0], "lon": kwargs.get("coordinates")[1]},
            "L": 1.0,
            "S": 1.0,
            "N": 1.0,
            "state": calculator.lookup_state_name_by_coordinates(kwargs.get("coordinates")[0], kwargs.get("coordinates")[1]),
            "color": "#000000"},
            new_team=True)
    current_popup = Popup(location=kwargs.get("coordinates"), min_width=420, max_width=420)
    current_popup.child = content
    map.add(current_popup)


for team in league["teams"]:
    if "color" in team:
       map.add(create_marker_for_team(team))

heatmap_counties(counties_geojson, result["county_stats_by_geoid"])
geojson_layer.data = counties_geojson 

geojson_layer.on_click(create_show_teams(map, result["county_stats_by_geoid"]))

map.add(geojson_layer)
map.add(FullScreenControl())
map.fullscreen = True

map.on_interaction(handle_interaction)

calc_button.on_click(recalculate_stats)

box = widgets.VBox([
    calc_button
])

control = WidgetControl(widget=box, position='topright')
map.add(control)

map

Elapsed: 7.390491243s


HTML(value='\n            <style>\n            .leaflet-popup-content-wrapper .leaflet-popup-tip {\n          …

Map(center=[38.72728229549864, -96.9010842308538], controls=(ZoomControl(options=['position', 'zoom_in_text', …

In [26]:
out

Output()